# ALL SET OF FEATURES SC M AND NM

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.tree import  DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import  SVR
from sklearn.neighbors import  KNeighborsRegressor

current_dir = Path.cwd()
project_root = current_dir.parents[2]

regression_models = {
    "decision_tree": DecisionTreeRegressor(random_state=42),
    "random_forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "gradient_boosting": XGBRegressor(device="cuda",tree_method="hist"),
    "ridge": Ridge(random_state=42),
    "lasso": Lasso(random_state=42),
    "svr": SVR(),
    "knn": KNeighborsRegressor(n_jobs=-1)
}

full_set_path_updrs3 = project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/REGRESSION/UPDRS3/FULL_SET/"
full_set_path_delta = project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/REGRESSION/DELTA_UPDRS3/FULL_SET/"

feature_selection_path_updrs3 = project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/REGRESSION/UPDRS3/FEATURE_SELECTION/"
feature_selection_path_delta = project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/MODEL_PERFORMANCE_RESULTS/REGRESSION/DELTA_UPDRS3/FEATURE_SELECTION/"

In [2]:
import json

with open(project_root/"SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/Domain_data.json", "r") as archivo:
    datos = json.load(archivo)

print(datos)

{'M_data': ['NP1COG_mean', 'NP1COG_min', 'NP1COG_max', 'NP1COG_var', 'NP1COG_std', 'NP1HALL_mean', 'NP1HALL_min', 'NP1HALL_max', 'NP1HALL_var', 'NP1HALL_std', 'NP1DPRS_mean', 'NP1DPRS_min', 'NP1DPRS_max', 'NP1DPRS_var', 'NP1DPRS_std', 'NP1ANXS_mean', 'NP1ANXS_min', 'NP1ANXS_max', 'NP1ANXS_var', 'NP1ANXS_std', 'NP1APAT_mean', 'NP1APAT_min', 'NP1APAT_max', 'NP1APAT_var', 'NP1APAT_std', 'NP1DDS_mean', 'NP1DDS_min', 'NP1DDS_max', 'NP1DDS_var', 'NP1DDS_std', 'NP1SLPN_mean', 'NP1SLPN_min', 'NP1SLPN_max', 'NP1SLPN_var', 'NP1SLPN_std', 'NP1SLPD_mean', 'NP1SLPD_min', 'NP1SLPD_max', 'NP1SLPD_var', 'NP1SLPD_std', 'NP1PAIN_mean', 'NP1PAIN_min', 'NP1PAIN_max', 'NP1PAIN_var', 'NP1PAIN_std', 'NP1URIN_mean', 'NP1URIN_min', 'NP1URIN_max', 'NP1URIN_var', 'NP1URIN_std', 'NP1CNST_mean', 'NP1CNST_min', 'NP1CNST_max', 'NP1CNST_var', 'NP1CNST_std', 'NP1LTHD_mean', 'NP1LTHD_min', 'NP1LTHD_max', 'NP1LTHD_var', 'NP1LTHD_std', 'NP1FATG_mean', 'NP1FATG_min', 'NP1FATG_max', 'NP1FATG_var', 'NP1FATG_std', 'NP2SPCH_m

In [3]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import KFold, ShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate_models_10x10_oof_and_test_regression(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.2,
    random_state: int = 42,
    decimals: int = 4,
):
    y = y_df.iloc[:, 0].to_numpy()
    X = X_df.to_numpy()

    def compute_metrics(y_true, y_pred):
        rmse = mean_squared_error(y_true, y_pred)
        return {
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": rmse,
            "R2": r2_score(y_true, y_pred),
        }

    def summarize(metrics_list, suffix: str):
        df = pd.DataFrame(metrics_list)
        mean = df.mean()
        std = df.std(ddof=1)
        return {
            f"{c}_{suffix}": f"{mean[c]:.{decimals}f} ± {std[c]:.{decimals}f}"
            for c in df.columns
        }

    outer = ShuffleSplit(
        n_splits=outer_splits, test_size=test_size, random_state=random_state
    )

    test_summary_rows = []
    cv_summary_rows = []

    for model_name, estimator in models.items():
        print(f"Evaluating {model_name}...")
        test_metrics_all = []
        cv_metrics_all = []

        for train_idx, test_idx in outer.split(X):
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]

            # ---------- INNER CV (OOF) ----------
            inner = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)

            oof_pred = np.empty_like(y_train, dtype=float)

            for tr_idx, val_idx in inner.split(X_train):
                pipe = Pipeline([
                    ("scaler", StandardScaler()),
                    ("model", clone(estimator)),
                ])
                pipe.fit(X_train[tr_idx], y_train[tr_idx])
                oof_pred[val_idx] = pipe.predict(X_train[val_idx])

            cv_metrics_all.append(compute_metrics(y_train, oof_pred))

            # ---------- TEST ----------
            pipe_full = Pipeline([
                ("scaler", StandardScaler()),
                ("model", clone(estimator)),
            ])
            pipe_full.fit(X_train, y_train)

            test_pred = pipe_full.predict(X_test)
            test_metrics_all.append(compute_metrics(y_test, test_pred))

        test_summary_rows.append(pd.Series(summarize(test_metrics_all, "Testing"), name=model_name))
        cv_summary_rows.append(pd.Series(summarize(cv_metrics_all, "CV"), name=model_name))

    df_test_summary = pd.DataFrame(test_summary_rows)[
        ["MAE_Testing", "RMSE_Testing", "R2_Testing"]
    ]
    df_cv_summary = pd.DataFrame(cv_summary_rows)[
        ["MAE_CV", "RMSE_CV", "R2_CV"]
    ]

    df_final_summary = pd.concat([df_test_summary, df_cv_summary], axis=1)
    return df_final_summary

## Delta UPDRS 3

In [4]:
X_delta_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_REG_full.csv", index_col=0)
y_delta_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_REG_full.csv", index_col=0)
print("X_delta_data shape:", X_delta_data.shape)
print("y_delta_data shape:", y_delta_data.shape)

X_delta_data shape: (909, 936)
y_delta_data shape: (909, 1)


In [5]:
delta_updrs=evaluate_models_10x10_oof_and_test_regression(
    X_df=X_delta_data,
    y_df=y_delta_data,
    models=regression_models)

delta_updrs.to_csv(full_set_path_delta / "DELTA_UPDRS3_REG_full_results.csv", index=True)
delta_updrs.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating gradient_boosting...


/home/fsc/Desktop/PD_progression_ppmi/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [14:04:09] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Evaluating ridge...
Evaluating lasso...
Evaluating svr...
Evaluating knn...


,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
decision_tree,6.3681 ± 0.5434,81.9505 ± 14.6070,-0.9768 ± 0.2885,6.7201 ± 0.2180,91.2928 ± 6.4266,-1.0303 ± 0.1350
random_forest,4.4247 ± 0.3371,41.6615 ± 6.6451,0.0017 ± 0.0412,4.7286 ± 0.0975,46.7018 ± 2.2604,-0.0379 ± 0.0221
gradient_boosting,4.8297 ± 0.3189,47.9324 ± 7.1026,-0.1522 ± 0.0848,5.1067 ± 0.1791,53.0357 ± 3.7131,-0.1781 ± 0.0514
ridge,11.1095 ± 0.8139,213.7801 ± 32.2316,-4.2793 ± 1.3946,10.9892 ± 0.3866,210.4815 ± 15.0649,-3.6813 ± 0.3273
lasso,4.4761 ± 0.3466,42.0879 ± 7.0743,-0.0062 ± 0.0149,4.6718 ± 0.0989,45.0517 ± 1.8003,-0.0014 ± 0.0022
svr,4.3907 ± 0.3698,42.2683 ± 7.1756,-0.0104 ± 0.0158,4.5736 ± 0.0925,44.9944 ± 1.8241,-0.0001 ± 0.0051
knn,4.7505 ± 0.3884,47.5031 ± 8.7468,-0.1351 ± 0.0693,4.8931 ± 0.1038,49.3318 ± 1.9920,-0.0967 ± 0.0198


## UPDRS 3

In [6]:
X_updrs3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_UPDRS3_REG_full.csv", index_col=0)
y_updrs3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_REG_full.csv", index_col=0)
print("X_updrs3_data shape:", X_updrs3_data.shape)
print("y_updrs3_data shape:", y_updrs3_data.shape)

X_updrs3_data shape: (909, 936)
y_updrs3_data shape: (909, 1)


In [7]:
updrs3_RESULTS=evaluate_models_10x10_oof_and_test_regression(
    X_df=X_updrs3_data,
    y_df=y_updrs3_data,
    models=regression_models)

updrs3_RESULTS.to_csv(full_set_path_updrs3 / "UPDRS3_REG_full_results.csv", index=True)
updrs3_RESULTS.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating gradient_boosting...
Evaluating ridge...
Evaluating lasso...
Evaluating svr...
Evaluating knn...


,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
decision_tree,7.3357 ± 0.5050,113.5720 ± 14.9739,0.5301 ± 0.0711,7.4069 ± 0.1727,115.6809 ± 6.8987,0.5347 ± 0.0293
random_forest,5.0978 ± 0.3892,53.4682 ± 9.3804,0.7808 ± 0.0291,5.1901 ± 0.1021,53.7609 ± 2.2117,0.7838 ± 0.0081
gradient_boosting,5.3739 ± 0.2826,60.5591 ± 7.2224,0.7510 ± 0.0210,5.4300 ± 0.1174,59.9050 ± 3.3329,0.7591 ± 0.0136
ridge,10.3864 ± 0.7365,193.8988 ± 34.4726,0.1982 ± 0.1569,10.3543 ± 0.4434,187.7352 ± 17.3748,0.2449 ± 0.0704
lasso,4.9293 ± 0.2770,46.5732 ± 6.5789,0.8086 ± 0.0207,4.9685 ± 0.0679,46.6662 ± 1.6504,0.8123 ± 0.0071
svr,7.3533 ± 0.3867,104.7280 ± 14.3918,0.5707 ± 0.0285,7.7271 ± 0.0906,111.0105 ± 3.1031,0.5536 ± 0.0095
knn,7.5819 ± 0.7361,120.7648 ± 22.8654,0.5040 ± 0.0846,7.8053 ± 0.1849,123.9562 ± 6.1247,0.5016 ± 0.0212


# SET OF FEATURES M

## Delta UPDRS 3

In [8]:
cols=datos['M_data']
X_delta_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_REG_full.csv", index_col=0)[cols]
y_delta_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_REG_full.csv", index_col=0)
print("X_delta_motor_data shape:", X_delta_motor_data.shape)
print("y_delta_motor_data shape:", y_delta_motor_data.shape)

X_delta_motor_data shape: (909, 305)
y_delta_motor_data shape: (909, 1)


In [ ]:
delta_motor_RESULTS=evaluate_models_10x10_oof_and_test_regression(
    X_df=X_delta_motor_data,
    y_df=y_delta_motor_data,
    models=regression_models)

delta_motor_RESULTS.to_csv(full_set_path_delta / "DELTA_UPDRS3_MOTOR_REG_full_results.csv", index=True)
delta_motor_RESULTS.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating gradient_boosting...
Evaluating ridge...
Evaluating lasso...
Evaluating svr...
Evaluating knn...


,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
decision_tree,6.6167 ± 0.6629,88.4893 ± 16.8678,-1.1375 ± 0.4010,6.6683 ± 0.1852,91.2082 ± 5.1410,-1.0280 ± 0.0960
random_forest,4.4871 ± 0.3265,42.6219 ± 7.1694,-0.0206 ± 0.0533,4.7599 ± 0.1214,47.5359 ± 2.3923,-0.0565 ± 0.0283
gradient_boosting,4.9043 ± 0.3033,50.4359 ± 7.0134,-0.2150 ± 0.0944,5.1791 ± 0.1111,54.9276 ± 2.3563,-0.2210 ± 0.0222
ridge,6.0580 ± 0.4823,76.4129 ± 12.6553,-0.8498 ± 0.2729,6.3934 ± 0.2105,85.4196 ± 7.9996,-0.8980 ± 0.1547
lasso,4.4761 ± 0.3466,42.0871 ± 7.0734,-0.0062 ± 0.0149,4.6714 ± 0.0990,45.0422 ± 1.8104,-0.0012 ± 0.0024
svr,4.3796 ± 0.3534,42.1375 ± 7.0359,-0.0078 ± 0.0183,4.5521 ± 0.0890,44.7731 ± 1.8062,0.0048 ± 0.0063
knn,4.7435 ± 0.3753,48.9022 ± 7.9439,-0.1733 ± 0.0914,4.8852 ± 0.1206,50.2976 ± 2.5161,-0.1178 ± 0.0262


## UPDRS 3

In [10]:
X_updrs3_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_UPDRS3_REG_full.csv", index_col=0)[cols]
y_updrs3_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_REG_full.csv", index_col=0)
print("X_updrs3_motor_data shape:", X_updrs3_motor_data.shape)
print("y_updrs3_motor_data shape:", y_updrs3_motor_data.shape)

X_updrs3_motor_data shape: (909, 305)
y_updrs3_motor_data shape: (909, 1)


In [ ]:
updrs3_motor_RESULTS=evaluate_models_10x10_oof_and_test_regression(
    X_df=X_updrs3_motor_data,
    y_df=y_updrs3_motor_data,
    models=regression_models)

updrs3_motor_RESULTS.to_csv(full_set_path_updrs3 / "UPDRS3_MOTOR_REG_full_results.csv", index=True)
updrs3_motor_RESULTS.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating gradient_boosting...
Evaluating ridge...
Evaluating lasso...
Evaluating svr...
Evaluating knn...


,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
decision_tree,7.1634 ± 0.5790,111.5458 ± 16.0524,0.5425 ± 0.0431,7.0189 ± 0.1989,103.5353 ± 4.3709,0.5836 ± 0.0178
random_forest,4.9696 ± 0.3501,51.3308 ± 8.5412,0.7895 ± 0.0263,5.0140 ± 0.1015,51.0810 ± 2.1334,0.7946 ± 0.0075
gradient_boosting,5.4359 ± 0.3345,62.7309 ± 7.2762,0.7415 ± 0.0272,5.3971 ± 0.1166,59.6029 ± 2.5112,0.7602 ± 0.0132
ridge,6.1203 ± 0.3884,76.3775 ± 9.9751,0.6839 ± 0.0491,6.3115 ± 0.1702,80.0233 ± 5.1741,0.6780 ± 0.0244
lasso,4.9348 ± 0.2767,46.6632 ± 6.5906,0.8082 ± 0.0208,4.9684 ± 0.0682,46.6450 ± 1.6495,0.8124 ± 0.0071
svr,5.8694 ± 0.3343,74.5140 ± 11.0620,0.6944 ± 0.0278,6.0870 ± 0.1000,75.9907 ± 2.7388,0.6944 ± 0.0092
knn,5.7649 ± 0.5081,71.3681 ± 13.1522,0.7078 ± 0.0404,5.8288 ± 0.1455,71.9446 ± 3.3262,0.7107 ± 0.0112


# SET OF FEATURES NM

## Delta UPDRS 3

In [12]:
cols=datos['NM_data']
X_delta_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_REG_full.csv", index_col=0)[cols]
y_delta_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_REG_full.csv", index_col=0)
print("X_delta_NO_motor_data shape:", X_delta_NO_motor_data.shape)
print("y_delta_NO_motor_data shape:", y_delta_NO_motor_data.shape)

X_delta_NO_motor_data shape: (909, 625)
y_delta_NO_motor_data shape: (909, 1)


In [ ]:
delta_NO_motor_RESULTS=evaluate_models_10x10_oof_and_test_regression(
    X_df=X_delta_NO_motor_data,
    y_df=y_delta_NO_motor_data,
    models=regression_models)

delta_NO_motor_RESULTS.to_csv(full_set_path_delta / "DELTA_UPDRS3_NO_MOTOR_REG_full_results.csv", index=True)
delta_NO_motor_RESULTS.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating gradient_boosting...
Evaluating ridge...
Evaluating lasso...
Evaluating svr...
Evaluating knn...


,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
decision_tree,7.1198 ± 0.4219,95.4209 ± 9.1692,-1.3341 ± 0.3991,7.3215 ± 0.3554,99.7952 ± 9.7224,-1.2218 ± 0.2343
random_forest,4.6417 ± 0.3784,44.2065 ± 7.5553,-0.0582 ± 0.0574,4.8532 ± 0.1270,47.5229 ± 2.2429,-0.0562 ± 0.0175
gradient_boosting,5.1039 ± 0.3294,49.4732 ± 7.0082,-0.1907 ± 0.0926,5.3719 ± 0.1408,54.4852 ± 3.4489,-0.2107 ± 0.0493
ridge,7.8920 ± 0.1784,105.8293 ± 6.1668,-1.5963 ± 0.4643,8.9418 ± 0.2723,134.8456 ± 10.4287,-1.9995 ± 0.2292
lasso,4.5055 ± 0.3460,42.4191 ± 7.1174,-0.0142 ± 0.0138,4.6887 ± 0.1042,45.0879 ± 1.7882,-0.0022 ± 0.0009
svr,4.4443 ± 0.3884,42.9798 ± 7.6595,-0.0261 ± 0.0301,4.6214 ± 0.0986,45.4649 ± 1.9431,-0.0105 ± 0.0062
knn,4.9368 ± 0.2862,48.1872 ± 6.7357,-0.1590 ± 0.0739,5.0737 ± 0.1145,49.3962 ± 2.0379,-0.0981 ± 0.0201


## UPDRS 3

In [14]:
X_updrs3_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_UPDRS3_REG_full.csv", index_col=0)[cols]
y_updrs3_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_REG_full.csv", index_col=0)
print("X_updrs3_NO_motor_data shape:", X_updrs3_NO_motor_data.shape)
print("y_updrs3_NO_motor_data shape:", y_updrs3_NO_motor_data.shape)

X_updrs3_NO_motor_data shape: (909, 625)
y_updrs3_NO_motor_data shape: (909, 1)


In [ ]:
updrs3_NO_motor_RESULTS=evaluate_models_10x10_oof_and_test_regression(
    X_df=X_updrs3_NO_motor_data,
    y_df=y_updrs3_NO_motor_data,
    models=regression_models)

updrs3_NO_motor_RESULTS.to_csv(full_set_path_updrs3 / "UPDRS3_NO_MOTOR_REG_full_results.csv", index=True)
updrs3_NO_motor_RESULTS.head(10)

Evaluating decision_tree...
Evaluating random_forest...
Evaluating gradient_boosting...
Evaluating ridge...
Evaluating lasso...
Evaluating svr...
Evaluating knn...


,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
decision_tree,14.6165 ± 1.0149,386.4912 ± 34.6782,-0.5960 ± 0.1675,14.8161 ± 0.7033,383.4978 ± 26.9238,-0.5419 ± 0.1016
random_forest,11.1087 ± 0.3930,186.5621 ± 16.7639,0.2320 ± 0.0438,11.4930 ± 0.1559,196.4544 ± 5.3953,0.2100 ± 0.0162
gradient_boosting,11.2335 ± 0.4424,204.2330 ± 16.4188,0.1574 ± 0.0688,11.5779 ± 0.3477,212.9413 ± 10.5636,0.1439 ± 0.0361
ridge,15.2036 ± 0.6956,398.5900 ± 34.6545,-0.6484 ± 0.1895,16.5585 ± 0.5848,465.3724 ± 43.0927,-0.8704 ± 0.1583
lasso,10.9995 ± 0.3709,177.4794 ± 15.8712,0.2697 ± 0.0351,11.2744 ± 0.1124,183.8857 ± 3.8812,0.2605 ± 0.0132
svr,11.9971 ± 0.6273,236.8647 ± 33.9610,0.0298 ± 0.0690,12.3294 ± 0.1480,245.4778 ± 3.8486,0.0128 ± 0.0130
knn,12.0612 ± 0.7987,243.3544 ± 35.4966,0.0027 ± 0.0880,12.2888 ± 0.1722,251.3599 ± 6.6773,-0.0107 ± 0.0196


# AGNOSTIC FEATURE SELECTION FULL 
- Correlation de spearman
- MI
- f_regression

In [16]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import KFold, ShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =========================================================
# Feature selectors para REGRESIÓN
# =========================================================

class SpearmanCorrelationDiscard(BaseEstimator, TransformerMixin):
    """
    Elimina variables altamente correlacionadas entre sí
    usando Spearman. De cada par altamente correlacionado,
    elimina la que tenga menor correlación absoluta con y.
    """

    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.vars_to_drop_ = None
        self.feature_names_in_ = None

    def fit(self, X, y=None):
        X = X.copy()

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.feature_names_in_ = X.columns.to_list()

        data = X.copy()
        data["_target_"] = y

        corr_df = data.corr(method="spearman")

        mask = np.tril(np.ones(corr_df.shape), k=-1).astype(bool)

        corr_long = (
            corr_df.where(mask)
            .stack()
            .reset_index()
            .rename(columns={"level_0": "V1", "level_1": "V2", 0: "CORR"})
        )

        corr_long = corr_long[
            (corr_long["V1"] != "_target_") & (corr_long["V2"] != "_target_")
        ].copy()

        target_corr = corr_df["_target_"]

        corr_long["V1target"] = corr_long["V1"].map(target_corr)
        corr_long["V2target"] = corr_long["V2"].map(target_corr)

        corr_long["WORST_VAR"] = np.where(
            abs(corr_long["V1target"]) <= abs(corr_long["V2target"]),
            corr_long["V1"],
            corr_long["V2"]
        )

        discard_corr_long = corr_long.loc[corr_long["CORR"].abs() > self.threshold]
        self.vars_to_drop_ = list(set(discard_corr_long["WORST_VAR"]))

        return self

    def transform(self, X):
        X = X.copy()

        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.vars_to_drop_, errors="ignore")

        X_df = pd.DataFrame(X, columns=self.feature_names_in_)
        X_df = X_df.drop(columns=self.vars_to_drop_, errors="ignore")
        return X_df


class MIRegressionThresholdSelector(BaseEstimator, TransformerMixin):
    """
    Selecciona variables con mutual_info_regression >= threshold.
    """

    def __init__(self, threshold=0.01, random_state=42):
        self.threshold = threshold
        self.random_state = random_state
        self.support_ = None
        self.mi_scores_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.mi_scores_ = mutual_info_regression(
            X_fit,
            y,
            random_state=self.random_state
        )
        self.support_ = self.mi_scores_ >= self.threshold

        if not np.any(self.support_):
            best_idx = np.argmax(self.mi_scores_)
            self.support_[best_idx] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


class FRegressionThresholdSelector(BaseEstimator, TransformerMixin):
    """
    Selecciona variables con f_regression >= threshold.
    """

    def __init__(self, threshold=1.0):
        self.threshold = threshold
        self.support_ = None
        self.f_scores_ = None
        self.pvalues_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.to_list()
            X_fit = X.values
        else:
            self.feature_names_in_ = None
            X_fit = X

        self.f_scores_, self.pvalues_ = f_regression(X_fit, y)
        self.support_ = self.f_scores_ >= self.threshold

        if not np.any(self.support_):
            best_idx = np.nanargmax(self.f_scores_)
            self.support_[best_idx] = True

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            cols = X.columns[np.asarray(self.support_)]
            return X.loc[:, cols]

        return X[:, self.support_]


# =========================================================
# Helpers
# =========================================================

def build_pipeline_regression(
    estimator,
    selector_name,
    spearman_threshold=0.9,
    mi_threshold=0.01,
    f_threshold=1.0,
    random_state=42,
):
    if selector_name == "spearman_corr":
        return Pipeline([
            ("selector", SpearmanCorrelationDiscard(threshold=spearman_threshold)),
            ("scaler", StandardScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "mutual_info":
        return Pipeline([
            ("selector", MIRegressionThresholdSelector(
                threshold=mi_threshold,
                random_state=random_state
            )),
            ("scaler", StandardScaler()),
            ("model", clone(estimator)),
        ])

    elif selector_name == "f_regression":
        return Pipeline([
            ("selector", FRegressionThresholdSelector(threshold=f_threshold)),
            ("scaler", StandardScaler()),
            ("model", clone(estimator)),
        ])

    else:
        raise ValueError(f"Selector no soportado: {selector_name}")


def compute_metrics_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred),
    }


def summarize(metrics_list, suffix="CV", decimals=4):
    df = pd.DataFrame(metrics_list)
    mean = df.mean(numeric_only=True)
    std = df.std(ddof=1, numeric_only=True)

    return {
        f"{c}_{suffix}": f"{mean[c]:.{decimals}f} ± {std[c]:.{decimals}f}"
        for c in mean.index
    }


# =========================================================
# Función principal REGRESIÓN con OOF + TEST + FS
# =========================================================

def evaluate_models_oof_and_test_with_fs_regression(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    models: dict,
    outer_splits: int = 10,
    inner_splits: int = 10,
    test_size: float = 0.2,
    random_state: int = 42,
    decimals: int = 4,
    selectors: list = None,
    spearman_threshold: float = 0.9,
    mi_threshold: float = 0.01,
    f_threshold: float = 1.0,
):
    """
    Evalúa múltiples modelos de regresión usando:
      - Outer CV: ShuffleSplit
      - Inner CV: KFold para OOF en train

    Para cada modelo y cada selector genera una fila.

    Selectores soportados:
      - "spearman_corr"
      - "mutual_info"
      - "f_regression"
    """

    if selectors is None:
        selectors = ["spearman_corr", "mutual_info", "f_regression"]

    X = X_df.copy()
    y = y_df.iloc[:, 0].to_numpy()

    outer = ShuffleSplit(
        n_splits=outer_splits,
        test_size=test_size,
        random_state=random_state
    )

    all_rows = []

    for model_name, estimator in models.items():
        for selector_name in selectors:
            print(f"Evaluating {model_name} + {selector_name}...")

            test_metrics_all = []
            cv_metrics_all = []

            for train_idx, test_idx in outer.split(X):
                X_train = X.iloc[train_idx].copy()
                X_test = X.iloc[test_idx].copy()
                y_train = y[train_idx]
                y_test = y[test_idx]

                # -----------------------------
                # OOF en TRAIN (inner CV)
                # -----------------------------
                inner = KFold(
                    n_splits=inner_splits,
                    shuffle=True,
                    random_state=random_state
                )

                oof_pred = np.empty_like(y_train, dtype=float)

                for tr_idx, val_idx in inner.split(X_train):
                    X_tr = X_train.iloc[tr_idx].copy()
                    X_val = X_train.iloc[val_idx].copy()
                    y_tr = y_train[tr_idx]

                    pipe = build_pipeline_regression(
                        estimator=estimator,
                        selector_name=selector_name,
                        spearman_threshold=spearman_threshold,
                        mi_threshold=mi_threshold,
                        f_threshold=f_threshold,
                        random_state=random_state,
                    )

                    pipe.fit(X_tr, y_tr)
                    oof_pred[val_idx] = pipe.predict(X_val)

                cv_metrics_all.append(compute_metrics_regression(y_train, oof_pred))

                # -----------------------------
                # TEST externo
                # -----------------------------
                pipe_full = build_pipeline_regression(
                    estimator=estimator,
                    selector_name=selector_name,
                    spearman_threshold=spearman_threshold,
                    mi_threshold=mi_threshold,
                    f_threshold=f_threshold,
                    random_state=random_state,
                )

                pipe_full.fit(X_train, y_train)
                test_pred = pipe_full.predict(X_test)

                test_metrics_all.append(compute_metrics_regression(y_test, test_pred))

            row = {
                "Model": model_name,
                "Feature_Selection": selector_name,
            }
            row.update(summarize(test_metrics_all, suffix="Testing", decimals=decimals))
            row.update(summarize(cv_metrics_all, suffix="CV", decimals=decimals))

            all_rows.append(row)

    df_final_summary = pd.DataFrame(all_rows)[[
        "Model",
        "Feature_Selection",
        "MAE_Testing",
        "RMSE_Testing",
        "R2_Testing",
        "MAE_CV",
        "RMSE_CV",
        "R2_CV",
    ]]

    return df_final_summary

## Delta UPDRS 3

In [17]:
X_delta_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_REG_full.csv", index_col=0)
y_delta_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_REG_full.csv", index_col=0)
print("X_delta_data shape:", X_delta_data.shape)
print("y_delta_data shape:", y_delta_data.shape)

X_delta_data shape: (909, 936)
y_delta_data shape: (909, 1)


In [18]:
delta_fe_reg_results = evaluate_models_oof_and_test_with_fs_regression(
    X_df=X_delta_data,
    y_df=y_delta_data,
    models=regression_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    selectors=["spearman_corr", "mutual_info", "f_regression"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    f_threshold=1.0,
)

delta_fe_reg_results.to_csv(feature_selection_path_delta / "DELTA_UPDRS3_FS_REG_full_results.csv", index=False)
delta_fe_reg_results.head(30)

Evaluating decision_tree + spearman_corr...
Evaluating decision_tree + mutual_info...
Evaluating decision_tree + f_regression...
Evaluating random_forest + spearman_corr...
Evaluating random_forest + mutual_info...
Evaluating random_forest + f_regression...
Evaluating gradient_boosting + spearman_corr...
Evaluating gradient_boosting + mutual_info...
Evaluating gradient_boosting + f_regression...
Evaluating ridge + spearman_corr...
Evaluating ridge + mutual_info...
Evaluating ridge + f_regression...
Evaluating lasso + spearman_corr...
Evaluating lasso + mutual_info...
Evaluating lasso + f_regression...
Evaluating svr + spearman_corr...
Evaluating svr + mutual_info...
Evaluating svr + f_regression...
Evaluating knn + spearman_corr...
Evaluating knn + mutual_info...
Evaluating knn + f_regression...


,Model,Feature_Selection,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
0,decision_tree,spearman_corr,6.5275 ± 0.5470,9.2328 ± 0.9000,-1.0795 ± 0.4055,6.8557 ± 0.2528,9.7579 ± 0.3934,-1.1177 ± 0.1139
1,decision_tree,mutual_info,6.5077 ± 0.5188,9.3078 ± 0.7307,-1.1066 ± 0.2908,6.8052 ± 0.2659,9.7271 ± 0.3878,-1.1058 ± 0.1419
2,decision_tree,f_regression,6.4973 ± 0.4018,9.1843 ± 0.6547,-1.0603 ± 0.3551,6.8528 ± 0.2649,9.8624 ± 0.4237,-1.1644 ± 0.1432
3,random_forest,spearman_corr,4.4110 ± 0.3751,6.3897 ± 0.5827,0.0169 ± 0.0532,4.7084 ± 0.1002,6.8002 ± 0.1607,-0.0283 ± 0.0206
4,random_forest,mutual_info,4.4300 ± 0.3274,6.4280 ± 0.5374,0.0043 ± 0.0428,4.7120 ± 0.1125,6.8102 ± 0.1798,-0.0312 ± 0.0225
5,random_forest,f_regression,4.4593 ± 0.3398,6.4844 ± 0.5512,-0.0130 ± 0.0454,4.7353 ± 0.0998,6.8526 ± 0.1477,-0.0442 ± 0.0178
6,gradient_boosting,spearman_corr,4.8397 ± 0.4005,6.9313 ± 0.5536,-0.1621 ± 0.1084,5.1310 ± 0.1534,7.2706 ± 0.2694,-0.1757 ± 0.0596
7,gradient_boosting,mutual_info,4.9066 ± 0.3479,7.0434 ± 0.5794,-0.1971 ± 0.0816,5.1347 ± 0.1062,7.2894 ± 0.1627,-0.1822 ± 0.0445
8,gradient_boosting,f_regression,4.8948 ± 0.3631,6.9748 ± 0.5681,-0.1765 ± 0.1110,5.1926 ± 0.1483,7.3745 ± 0.2276,-0.2096 ± 0.0513
9,ridge,spearman_corr,8.2209 ± 0.4877,10.8124 ± 0.6795,-1.8801 ± 0.6506,8.7656 ± 0.3178,11.6337 ± 0.4124,-2.0133 ± 0.2078


## UPDRS 3

In [19]:
X_updrs3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_UPDRS3_REG_full.csv", index_col=0)
y_updrs3_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_REG_full.csv", index_col=0)
print("X_updrs3_data shape:", X_updrs3_data.shape)
print("y_updrs3_data shape:", y_updrs3_data.shape)

X_updrs3_data shape: (909, 936)
y_updrs3_data shape: (909, 1)


In [20]:
UPDRS3_fe_reg_results = evaluate_models_oof_and_test_with_fs_regression(
    X_df=X_updrs3_data,
    y_df=y_updrs3_data,
    models=regression_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    selectors=["spearman_corr", "mutual_info", "f_regression"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    f_threshold=1.0,
)

UPDRS3_fe_reg_results.to_csv(feature_selection_path_updrs3 / "UPDRS3_FS_REG_full_results.csv", index=False)
UPDRS3_fe_reg_results.head(30)

Evaluating decision_tree + spearman_corr...
Evaluating decision_tree + mutual_info...
Evaluating decision_tree + f_regression...
Evaluating random_forest + spearman_corr...
Evaluating random_forest + mutual_info...
Evaluating random_forest + f_regression...
Evaluating gradient_boosting + spearman_corr...
Evaluating gradient_boosting + mutual_info...
Evaluating gradient_boosting + f_regression...
Evaluating ridge + spearman_corr...
Evaluating ridge + mutual_info...
Evaluating ridge + f_regression...
Evaluating lasso + spearman_corr...
Evaluating lasso + mutual_info...
Evaluating lasso + f_regression...
Evaluating svr + spearman_corr...
Evaluating svr + mutual_info...
Evaluating svr + f_regression...
Evaluating knn + spearman_corr...
Evaluating knn + mutual_info...
Evaluating knn + f_regression...


,Model,Feature_Selection,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
0,decision_tree,spearman_corr,7.3033 ± 0.7445,10.7053 ± 1.2263,0.5231 ± 0.1042,7.4862 ± 0.1890,10.8306 ± 0.3162,0.5279 ± 0.0270
1,decision_tree,mutual_info,7.3044 ± 0.4456,10.7330 ± 0.6566,0.5246 ± 0.0449,7.2541 ± 0.2251,10.4727 ± 0.2976,0.5584 ± 0.0276
2,decision_tree,f_regression,7.3890 ± 0.4727,10.7408 ± 0.4649,0.5236 ± 0.0358,7.2994 ± 0.2484,10.5703 ± 0.3488,0.5501 ± 0.0308
3,random_forest,spearman_corr,5.0431 ± 0.3880,7.2085 ± 0.6387,0.7854 ± 0.0287,5.1731 ± 0.1006,7.3074 ± 0.1483,0.7852 ± 0.0078
4,random_forest,mutual_info,5.0475 ± 0.3784,7.2124 ± 0.6412,0.7853 ± 0.0281,5.1301 ± 0.0791,7.2555 ± 0.1388,0.7882 ± 0.0081
5,random_forest,f_regression,5.0796 ± 0.3974,7.2601 ± 0.6666,0.7823 ± 0.0301,5.1612 ± 0.1011,7.2944 ± 0.1420,0.7860 ± 0.0078
6,gradient_boosting,spearman_corr,5.2521 ± 0.4486,7.6551 ± 0.6539,0.7580 ± 0.0312,5.4306 ± 0.1505,7.7146 ± 0.2022,0.7605 ± 0.0123
7,gradient_boosting,mutual_info,5.3772 ± 0.3820,7.7629 ± 0.5341,0.7515 ± 0.0230,5.4275 ± 0.1135,7.7178 ± 0.1903,0.7602 ± 0.0135
8,gradient_boosting,f_regression,5.3257 ± 0.3034,7.6279 ± 0.4021,0.7597 ± 0.0219,5.4598 ± 0.0964,7.7550 ± 0.1637,0.7580 ± 0.0115
9,ridge,spearman_corr,8.2098 ± 0.5181,10.9242 ± 0.6196,0.5038 ± 0.0769,8.6742 ± 0.3704,11.5389 ± 0.4674,0.4638 ± 0.0422


# AGNOSTIC FEATURE SELECTION MOTOR
- Correlation de spearman
- MI
- f_regression

## Delta UPDRS 3

In [21]:
cols=datos['M_data']
X_delta_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_REG_full.csv", index_col=0)[cols]
y_delta_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_REG_full.csv", index_col=0)
print("X_delta_motor_data shape:", X_delta_motor_data.shape)
print("y_delta_motor_data shape:", y_delta_motor_data.shape)

X_delta_motor_data shape: (909, 305)
y_delta_motor_data shape: (909, 1)


In [22]:
delta_fe_motor_reg_results = evaluate_models_oof_and_test_with_fs_regression(
    X_df=X_delta_motor_data,
    y_df=y_delta_motor_data,
    models=regression_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    selectors=["spearman_corr", "mutual_info", "f_regression"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    f_threshold=1.0,
)
delta_fe_motor_reg_results.to_csv(feature_selection_path_delta / "DELTA_UPDRS3_MOTOR_FS_REG_full_results.csv", index=False)
delta_fe_motor_reg_results.head(30)

Evaluating decision_tree + spearman_corr...
Evaluating decision_tree + mutual_info...
Evaluating decision_tree + f_regression...
Evaluating random_forest + spearman_corr...
Evaluating random_forest + mutual_info...
Evaluating random_forest + f_regression...
Evaluating gradient_boosting + spearman_corr...
Evaluating gradient_boosting + mutual_info...
Evaluating gradient_boosting + f_regression...
Evaluating ridge + spearman_corr...
Evaluating ridge + mutual_info...
Evaluating ridge + f_regression...
Evaluating lasso + spearman_corr...
Evaluating lasso + mutual_info...
Evaluating lasso + f_regression...
Evaluating svr + spearman_corr...
Evaluating svr + mutual_info...
Evaluating svr + f_regression...
Evaluating knn + spearman_corr...
Evaluating knn + mutual_info...
Evaluating knn + f_regression...


,Model,Feature_Selection,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
0,decision_tree,spearman_corr,6.5345 ± 0.5453,9.3529 ± 0.7084,-1.1278 ± 0.3092,6.7903 ± 0.2670,9.7671 ± 0.4032,-1.1227 ± 0.1359
1,decision_tree,mutual_info,6.5037 ± 0.5668,9.1093 ± 0.7694,-1.0164 ± 0.2839,6.6600 ± 0.2055,9.5551 ± 0.3855,-1.0310 ± 0.1184
2,decision_tree,f_regression,6.7132 ± 0.5230,9.6253 ± 0.6820,-1.2658 ± 0.4053,6.8531 ± 0.2509,9.8610 ± 0.4486,-1.1641 ± 0.1585
3,random_forest,spearman_corr,4.4823 ± 0.3257,6.4983 ± 0.5493,-0.0180 ± 0.0562,4.7480 ± 0.1219,6.8738 ± 0.1889,-0.0507 ± 0.0305
4,random_forest,mutual_info,4.4576 ± 0.3018,6.4519 ± 0.5336,-0.0034 ± 0.0480,4.7591 ± 0.1295,6.8731 ± 0.1886,-0.0505 ± 0.0291
5,random_forest,f_regression,4.5251 ± 0.3247,6.5329 ± 0.5096,-0.0294 ± 0.0489,4.7906 ± 0.1140,6.9423 ± 0.1692,-0.0717 ± 0.0226
6,gradient_boosting,spearman_corr,4.8713 ± 0.2186,7.0373 ± 0.3820,-0.2058 ± 0.1553,5.1627 ± 0.1115,7.3793 ± 0.1704,-0.2111 ± 0.0325
7,gradient_boosting,mutual_info,4.8790 ± 0.2884,7.0905 ± 0.4187,-0.2216 ± 0.1413,5.1992 ± 0.0852,7.4514 ± 0.1668,-0.2352 ± 0.0435
8,gradient_boosting,f_regression,4.9728 ± 0.2601,7.1948 ± 0.4784,-0.2522 ± 0.0919,5.1940 ± 0.1384,7.4847 ± 0.1763,-0.2463 ± 0.0451
9,ridge,spearman_corr,5.4222 ± 0.3738,7.8508 ± 0.7064,-0.4953 ± 0.2023,5.6426 ± 0.1657,8.1056 ± 0.3244,-0.4621 ± 0.0954


## UPDRS 3

In [23]:
X_updrs3_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_UPDRS3_REG_full.csv", index_col=0)[cols]
y_updrs3_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_REG_full.csv", index_col=0)
print("X_updrs3_motor_data shape:", X_updrs3_motor_data.shape)
print("y_updrs3_motor_data shape:", y_updrs3_motor_data.shape)

X_updrs3_motor_data shape: (909, 305)
y_updrs3_motor_data shape: (909, 1)


In [24]:
UPDRS_fe_motor_reg_results = evaluate_models_oof_and_test_with_fs_regression(
    X_df=X_updrs3_motor_data,
    y_df=y_updrs3_motor_data,
    models=regression_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    selectors=["spearman_corr", "mutual_info", "f_regression"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    f_threshold=1.0,
)

UPDRS_fe_motor_reg_results.to_csv(feature_selection_path_updrs3 / "UPDRS3_MOTOR_FS_REG_full_results.csv", index=False)
UPDRS_fe_motor_reg_results.head(30)

Evaluating decision_tree + spearman_corr...
Evaluating decision_tree + mutual_info...
Evaluating decision_tree + f_regression...
Evaluating random_forest + spearman_corr...
Evaluating random_forest + mutual_info...
Evaluating random_forest + f_regression...
Evaluating gradient_boosting + spearman_corr...
Evaluating gradient_boosting + mutual_info...
Evaluating gradient_boosting + f_regression...
Evaluating ridge + spearman_corr...
Evaluating ridge + mutual_info...
Evaluating ridge + f_regression...
Evaluating lasso + spearman_corr...
Evaluating lasso + mutual_info...
Evaluating lasso + f_regression...
Evaluating svr + spearman_corr...
Evaluating svr + mutual_info...
Evaluating svr + f_regression...
Evaluating knn + spearman_corr...
Evaluating knn + mutual_info...
Evaluating knn + f_regression...


,Model,Feature_Selection,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
0,decision_tree,spearman_corr,6.9739 ± 0.3983,10.2216 ± 0.7061,0.5675 ± 0.0551,7.2356 ± 0.2867,10.4152 ± 0.4541,0.5630 ± 0.0372
1,decision_tree,mutual_info,7.1298 ± 0.5536,10.5000 ± 0.7897,0.5464 ± 0.0334,7.0019 ± 0.2610,10.0973 ± 0.2931,0.5897 ± 0.0228
2,decision_tree,f_regression,7.0502 ± 0.7784,10.3997 ± 0.9688,0.5538 ± 0.0594,7.0558 ± 0.1934,10.2408 ± 0.1576,0.5782 ± 0.0119
3,random_forest,spearman_corr,4.9035 ± 0.3538,7.0531 ± 0.6208,0.7946 ± 0.0267,5.0074 ± 0.0952,7.1231 ± 0.1453,0.7959 ± 0.0071
4,random_forest,mutual_info,4.9585 ± 0.3398,7.1149 ± 0.5916,0.7910 ± 0.0263,4.9981 ± 0.1095,7.1277 ± 0.1537,0.7956 ± 0.0079
5,random_forest,f_regression,4.9515 ± 0.3457,7.1199 ± 0.6082,0.7907 ± 0.0271,5.0043 ± 0.0977,7.1395 ± 0.1461,0.7950 ± 0.0076
6,gradient_boosting,spearman_corr,5.3586 ± 0.3958,7.7381 ± 0.5880,0.7525 ± 0.0297,5.3459 ± 0.0896,7.6502 ± 0.1352,0.7645 ± 0.0092
7,gradient_boosting,mutual_info,5.3946 ± 0.3419,7.8061 ± 0.4561,0.7481 ± 0.0260,5.4194 ± 0.1177,7.7499 ± 0.1887,0.7583 ± 0.0126
8,gradient_boosting,f_regression,5.4185 ± 0.3970,7.8377 ± 0.5144,0.7461 ± 0.0280,5.3910 ± 0.1340,7.7012 ± 0.1960,0.7612 ± 0.0150
9,ridge,spearman_corr,5.5133 ± 0.2843,7.7973 ± 0.4501,0.7487 ± 0.0261,5.4865 ± 0.1518,7.7046 ± 0.2093,0.7611 ± 0.0137


# AGNOSTIC FEATURE SELECTION NON MOTOR
- Correlation de spearman
- MI
- f_regression

## Delta UPDRS 3

In [25]:
cols=datos['NM_data']
X_delta_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_MCID_REG_full.csv", index_col=0)[cols]
y_delta_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_REG_full.csv", index_col=0)
print("X_delta_NO_motor_data shape:", X_delta_NO_motor_data.shape)
print("y_delta_NO_motor_data shape:", y_delta_NO_motor_data.shape)

X_delta_NO_motor_data shape: (909, 625)
y_delta_NO_motor_data shape: (909, 1)


In [26]:
delta_fe_NO_motor_reg_results = evaluate_models_oof_and_test_with_fs_regression(
    X_df=X_delta_NO_motor_data,
    y_df=y_delta_NO_motor_data,
    models=regression_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    selectors=["spearman_corr", "mutual_info", "f_regression"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    f_threshold=1.0,
)
delta_fe_NO_motor_reg_results.to_csv(feature_selection_path_delta / "DELTA_UPDRS3_NO_MOTOR_FS_REG_full_results.csv", index=False)
delta_fe_NO_motor_reg_results.head(30)

Evaluating decision_tree + spearman_corr...
Evaluating decision_tree + mutual_info...
Evaluating decision_tree + f_regression...
Evaluating random_forest + spearman_corr...
Evaluating random_forest + mutual_info...
Evaluating random_forest + f_regression...
Evaluating gradient_boosting + spearman_corr...
Evaluating gradient_boosting + mutual_info...
Evaluating gradient_boosting + f_regression...
Evaluating ridge + spearman_corr...
Evaluating ridge + mutual_info...
Evaluating ridge + f_regression...
Evaluating lasso + spearman_corr...
Evaluating lasso + mutual_info...
Evaluating lasso + f_regression...
Evaluating svr + spearman_corr...
Evaluating svr + mutual_info...
Evaluating svr + f_regression...
Evaluating knn + spearman_corr...
Evaluating knn + mutual_info...
Evaluating knn + f_regression...


,Model,Feature_Selection,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
0,decision_tree,spearman_corr,6.9082 ± 0.4356,9.4587 ± 0.5908,-1.1825 ± 0.3314,7.2398 ± 0.2907,9.8916 ± 0.3938,-1.1782 ± 0.1542
1,decision_tree,mutual_info,7.2302 ± 0.3995,9.9290 ± 0.3759,-1.4321 ± 0.5037,7.1681 ± 0.2109,9.7892 ± 0.1956,-1.1318 ± 0.0676
2,decision_tree,f_regression,7.1462 ± 0.4187,9.8217 ± 0.6681,-1.3623 ± 0.4331,7.2202 ± 0.1961,9.8574 ± 0.2492,-1.1614 ± 0.0767
3,random_forest,spearman_corr,4.6659 ± 0.4089,6.6375 ± 0.6061,-0.0607 ± 0.0573,4.8605 ± 0.1073,6.8875 ± 0.1447,-0.0549 ± 0.0149
4,random_forest,mutual_info,4.6286 ± 0.3949,6.6227 ± 0.5928,-0.0559 ± 0.0493,4.8519 ± 0.1033,6.8713 ± 0.1552,-0.0500 ± 0.0212
5,random_forest,f_regression,4.7197 ± 0.3932,6.6859 ± 0.5741,-0.0768 ± 0.0477,4.9019 ± 0.1016,6.9344 ± 0.1477,-0.0694 ± 0.0186
6,gradient_boosting,spearman_corr,5.2116 ± 0.2932,7.1491 ± 0.4862,-0.2378 ± 0.1099,5.3674 ± 0.1842,7.3587 ± 0.2668,-0.2040 ± 0.0482
7,gradient_boosting,mutual_info,5.2372 ± 0.3866,7.1853 ± 0.5283,-0.2493 ± 0.1150,5.4023 ± 0.1530,7.4071 ± 0.1987,-0.2203 ± 0.0414
8,gradient_boosting,f_regression,5.2590 ± 0.4065,7.1877 ± 0.5751,-0.2460 ± 0.0642,5.4431 ± 0.1743,7.5081 ± 0.2469,-0.2537 ± 0.0534
9,ridge,spearman_corr,6.6884 ± 0.4938,8.7795 ± 0.5624,-0.8756 ± 0.2546,7.1457 ± 0.2002,9.3707 ± 0.2951,-0.9539 ± 0.0999


## UPDRS 3

In [27]:
X_updrs3_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/X_UPDRS3_REG_full.csv", index_col=0)[cols]
y_updrs3_NO_motor_data = pd.read_csv(project_root / "SCRIPTS/PYTHON/MODEL ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_UPDRS3_REG_full.csv", index_col=0)
print("X_updrs3_NO_motor_data shape:", X_updrs3_NO_motor_data.shape)
print("y_updrs3_NO_motor_data shape:", y_updrs3_NO_motor_data.shape)

X_updrs3_NO_motor_data shape: (909, 625)
y_updrs3_NO_motor_data shape: (909, 1)


In [28]:
UPDRS_fe_NO_motor_reg_results = evaluate_models_oof_and_test_with_fs_regression(
    X_df=X_updrs3_NO_motor_data,
    y_df=y_updrs3_NO_motor_data,
    models=regression_models,
    outer_splits=10,
    inner_splits=10,
    test_size=0.2,
    random_state=42,
    selectors=["spearman_corr", "mutual_info", "f_regression"],
    spearman_threshold=0.9,
    mi_threshold=0.01,
    f_threshold=1.0,
)

UPDRS_fe_NO_motor_reg_results.to_csv(feature_selection_path_updrs3 / "UPDRS3_NO_MOTOR_FS_REG_full_results.csv", index=False)
UPDRS_fe_NO_motor_reg_results.head(30)

Evaluating decision_tree + spearman_corr...
Evaluating decision_tree + mutual_info...
Evaluating decision_tree + f_regression...
Evaluating random_forest + spearman_corr...
Evaluating random_forest + mutual_info...
Evaluating random_forest + f_regression...
Evaluating gradient_boosting + spearman_corr...
Evaluating gradient_boosting + mutual_info...
Evaluating gradient_boosting + f_regression...
Evaluating ridge + spearman_corr...
Evaluating ridge + mutual_info...
Evaluating ridge + f_regression...
Evaluating lasso + spearman_corr...
Evaluating lasso + mutual_info...
Evaluating lasso + f_regression...
Evaluating svr + spearman_corr...
Evaluating svr + mutual_info...
Evaluating svr + f_regression...
Evaluating knn + spearman_corr...
Evaluating knn + mutual_info...
Evaluating knn + f_regression...


,Model,Feature_Selection,MAE_Testing,RMSE_Testing,R2_Testing,MAE_CV,RMSE_CV,R2_CV
0,decision_tree,spearman_corr,14.2225 ± 0.9364,18.9807 ± 1.2130,-0.4857 ± 0.1247,15.0420 ± 0.4449,19.7775 ± 0.4570,-0.5738 ± 0.0707
1,decision_tree,mutual_info,14.4363 ± 0.7526,19.1755 ± 1.0075,-0.5187 ± 0.1382,14.8898 ± 0.6727,19.6787 ± 0.7632,-0.5591 ± 0.1126
2,decision_tree,f_regression,14.9077 ± 0.7568,19.7767 ± 0.8773,-0.6168 ± 0.1476,14.6678 ± 0.5576,19.4477 ± 0.6441,-0.5214 ± 0.0790
3,random_forest,spearman_corr,11.1365 ± 0.4611,13.6739 ± 0.6622,0.2287 ± 0.0531,11.5153 ± 0.1886,14.0297 ± 0.2126,0.2083 ± 0.0202
4,random_forest,mutual_info,11.0453 ± 0.4352,13.5791 ± 0.6299,0.2398 ± 0.0409,11.5009 ± 0.2159,14.0089 ± 0.2184,0.2106 ± 0.0229
5,random_forest,f_regression,11.0705 ± 0.4687,13.6313 ± 0.6265,0.2337 ± 0.0456,11.4809 ± 0.1585,14.0116 ± 0.1789,0.2104 ± 0.0175
6,gradient_boosting,spearman_corr,11.1520 ± 0.5933,14.1989 ± 0.6912,0.1681 ± 0.0599,11.6084 ± 0.3977,14.6438 ± 0.3875,0.1374 ± 0.0381
7,gradient_boosting,mutual_info,11.1182 ± 0.5586,14.0910 ± 0.7704,0.1803 ± 0.0701,11.6206 ± 0.2021,14.6436 ± 0.2970,0.1375 ± 0.0276
8,gradient_boosting,f_regression,11.4345 ± 0.6060,14.4216 ± 0.8135,0.1413 ± 0.0788,11.6384 ± 0.2655,14.6156 ± 0.2382,0.1408 ± 0.0232
9,ridge,spearman_corr,13.2180 ± 0.6528,17.5815 ± 0.9976,-0.2818 ± 0.1799,13.9403 ± 0.3156,18.0247 ± 0.5803,-0.3071 ± 0.0705
